In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("luerhard/PopBERT")

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(
    "manifesto-project/manifestoberta-xlm-roberta-56policy-topics-context-2023-1-1",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-large")

sentence = "These principles are under threat."
context = "Human rights and international humanitarian law are fundamental pillars of a secure global system. These principles are under threat. Some of the world's most powerful states choose to sell arms to human-rights abusing states."
# For sentences without additional context, just use the sentence itself as the context.
# Example: context = "These principles are under threat."

In [ ]:
def flatten_list(li):
    if isinstance(li, list):
        return sum(map(flatten_list, li), [])
    return [li]

In [ ]:
from src.data.processors import TranscriptCleaner

cleaner = TranscriptCleaner()

sentence = "These principles are under threat."
context = "Human rights and international humanitarian law are fundamental pillars of a secure global system. And i love sports"

sent_tokens = flatten_list(cleaner.tokenize(sentence))
context_tokens = flatten_list(cleaner.tokenize(context))

print(sent_tokens)
print(context_tokens)

In [ ]:
# For sentences without additional context, just use the sentence itself as the context.
# Example: context = "These principles are under threat."

inputs = tokenizer(
    [sentence],
    [context],
    is_split_into_words=True,
    return_tensors="pt",
    max_length=300,  # we limited the input to 300 tokens during finetuning
    padding="max_length",
    truncation=True,
)

logits = model(**inputs).logits

probabilities = torch.softmax(logits, dim=1).tolist()[0]
probabilities = {
    model.config.id2label[index]: round(probability * 100, 2)
    for index, probability in enumerate(probabilities)
}
probabilities = dict(sorted(probabilities.items(), key=lambda item: item[1], reverse=True))
print(probabilities)
# {'201 - Freedom and Human Rights': 90.76, '107 - Internationalism: Positive': 5.82, '105 - Military: Negative': 0.66...

predicted_class = model.config.id2label[logits.argmax().item()]
print(predicted_class)
# 201 - Freedom and Human Rights

In [ ]:
tokens = cleaner.tokenize(sentence + sentence)
context_tokens = cleaner.tokenize(context + sentence)
print(tokens)
print(context_tokens)

inputs = tokenizer(
    tokens,
    context_tokens,
    is_split_into_words=True,
    return_tensors="pt",
    max_length=300,
    padding="max_length",
    truncation=True,
)

with torch.inference_mode():
    out = model(**inputs)

In [ ]:
import numpy as np

In [ ]:
labels = model.config.id2label

In [ ]:
probabilities = torch.softmax(out.logits, dim=1).detach().cpu().numpy()
preds = np.argmax(probabilities, axis=1)

In [ ]:
[labels[i] for i in preds]

In [ ]:
probabilities = torch.softmax(logits, dim=1).tolist()[0]
probabilities = {
    model.config.id2label[index]: round(probability * 100, 2)
    for index, probability in enumerate(probabilities)
}
probabilities = dict(sorted(probabilities.items(), key=lambda item: item[1], reverse=True))
print(probabilities)
# {'201 - Freedom and Human Rights': 90.76, '107 - Internationalism: Positive': 5.82, '105 - Military: Negative': 0.66...

predicted_class = model.config.id2label[logits.argmax().item()]
print(predicted_class)
# 201 - Freedom and Human Rights